# 04 自治体戦略タイプの分類

楽天データ（商品数・価格・レビュー）と総務省データ（受入額）を組み合わせ、
K-means クラスタリングで自治体を **5つの戦略タイプ** に分類する。

**分析の問い**
- 特産品（食品）依存型 vs 家電・汎用品型の自治体は分かれるか？
- 楽天での存在感（商品数・レビュー）と総務省の受入額は相関するか？
- 同じカテゴリ内でも自治体の価格戦略に差はあるか？

In [1]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from pathlib import Path

from src.analysis.municipality import build_shop_features, merge_soumu, run_kmeans, label_clusters
from src.api.soumu_loader import load_soumu_csv

pd.set_option('display.float_format', '{:,.1f}'.format)
DATA_DIR = Path('../data')

df = pd.read_parquet(DATA_DIR / 'processed/products.parquet')
df_soumu = load_soumu_csv(DATA_DIR / 'raw/soumu_donations.csv')
print(f'楽天: {len(df):,}件 / 総務省: {len(df_soumu):,}自治体')

2026-05-07 14:24:36 | INFO     | src.api.soumu_loader:56 - CSVロード: 1890 行 × 33 列
2026-05-07 14:24:36 | INFO     | src.api.soumu_loader:89 - 総務省データ読み込み完了 (令和４年度): 1,741 自治体


楽天: 32,244件 / 総務省: 1,741自治体


## 1. ショップ（自治体）別特徴量の作成

In [2]:
df_shops = build_shop_features(df)
print(f'楽天出店自治体数: {len(df_shops):,}')
df_shops.describe()

楽天出店自治体数: 1,497


,商品数,価格_中央値,価格_平均,レビュー数_合計,レビュー数_平均,評価平均,食品数,食品比率,高額品数,高額品比率
count,"1,497.0","1,497.0","1,497.0","1,497.0","1,497.0","1,380.0","1,497.0","1,497.0","1,497.0","1,497.0"
mean,21.5,"33,542.8","57,597.2",736.8,19.4,4.5,14.4,0.6,5.5,0.3
std,36.5,"109,601.7","152,229.1","3,383.4",51.6,0.4,28.8,0.3,12.8,0.3
min,1.0,"2,500.0","2,500.0",0.0,0.0,1.0,0.0,0.0,0.0,0.0
25%,4.0,"12,000.0","15,000.0",8.0,1.7,4.4,2.0,0.3,0.0,0.0
50%,10.0,"15,000.0","22,756.8",51.0,5.7,4.6,5.0,0.7,2.0,0.2
75%,25.0,"22,000.0","44,350.0",352.0,16.4,4.8,15.0,0.9,6.0,0.4
max,565.0,"2,039,000.0","2,704,250.0","67,999.0","1,010.9",5.0,500.0,1.0,233.0,1.0


In [3]:
# 主力カテゴリ分布
print('主力カテゴリ別 自治体数:')
print(df_shops['主力カテゴリ'].value_counts())

主力カテゴリ別 自治体数:
主力カテゴリ
魚介類         223
牛肉          216
果物          188
その他         185
旅行・体験       150
日用品・生活雑貨    107
米・穀物        100
鶏肉           93
野菜           69
家電・電気製品      69
酒・飲料         56
豚肉           34
加工食品・惣菜       6
スイーツ・菓子       1
Name: count, dtype: int64


## 2. 総務省データとの突合

In [4]:
df_merged, match_rate = merge_soumu(df_shops, df_soumu)
print(f'突合率: {match_rate:.1f}%')
print(f'突合成功: {df_merged["都道府県名"].notna().sum():,} / {len(df_merged):,} 自治体')
df_merged[df_merged['都道府県名'].notna()].head(5)[['shop_name','都道府県名','市区町村名','商品数','受入額_億円']]

突合率: 99.5%
突合成功: 1,489 / 1,497 自治体


,shop_name,都道府県名,市区町村名,商品数,受入額_億円
0,三重県いなべ市,三重県,いなべ市,3,0.9
1,三重県亀山市,三重県,亀山市,12,0.3
2,三重県伊勢市,三重県,伊勢市,20,4.6
3,三重県伊賀市,三重県,伊賀市,9,6.4
4,三重県南伊勢町,三重県,南伊勢町,9,1.6


In [6]:
# 楽天商品数 vs 総務省受入額の相関確認
df_both = df_merged[df_merged['受入額_億円'] > 0].copy()

fig = px.scatter(
    df_both,
    x='商品数', y='受入額_億円',
    color='主力カテゴリ',
    hover_data=['shop_name'],
    title='楽天 出品数 vs 総務省 受入額（令和4年度）',
    labels={'商品数': '楽天 出品商品数', '受入額_億円': 'ふるさと納税 受入額（億円）'},
    trendline='ols',
    trendline_scope='overall',
    opacity=0.7,
    height=500
)
fig.show()

corr = df_both[['商品数','受入額_億円','レビュー数_合計']].corr()
print('相関係数:')
print(corr)

ModuleNotFoundError: No module named 'statsmodels'

## 3. K-means クラスタリング

In [ ]:
# エルボー法でクラスタ数を確認
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from src.analysis.municipality import FEATURE_COLS

feature_cols = [c for c in FEATURE_COLS if c in df_merged.columns]
X = df_merged[feature_cols].fillna(0).values
X_scaled = StandardScaler().fit_transform(X)

inertias = []
for k in range(2, 10):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_scaled)
    inertias.append(km.inertia_)

fig = px.line(
    x=list(range(2, 10)), y=inertias,
    markers=True,
    title='エルボー法：最適クラスタ数の確認',
    labels={'x': 'クラスタ数', 'y': 'Inertia（低いほど密なクラスタ）'}
)
fig.show()

In [ ]:
# 5クラスタで実行
df_clustered, km_model, scaler = run_kmeans(df_merged, n_clusters=5)
df_clustered, cluster_labels = label_clusters(df_clustered)

print('クラスタ別 自治体数:')
print(df_clustered.groupby(['cluster', 'cluster_label']).size().reset_index(name='自治体数'))

In [ ]:
# クラスタ別 特徴量平均
cluster_profile = df_clustered.groupby('cluster_label')[feature_cols].mean().round(1)
cluster_profile

## 4. クラスタの可視化

In [ ]:
# 価格中央値 × レビュー数合計（クラスタ色分け）
fig = px.scatter(
    df_clustered,
    x='価格_中央値', y='レビュー数_合計',
    color='cluster_label',
    size='商品数',
    hover_data=['shop_name', '主力カテゴリ', '受入額_億円'],
    title='自治体 戦略タイプ別ポジショニング<br><sub>横: 価格中央値 / 縦: レビュー数合計 / バブル: 商品数</sub>',
    labels={'価格_中央値': '価格中央値（円）', 'レビュー数_合計': 'レビュー数合計'},
    color_discrete_sequence=px.colors.qualitative.Set1,
    opacity=0.7,
    height=550
)
fig.show()

In [ ]:
# 食品比率 × 高額品比率（クラスタ色分け）
fig = px.scatter(
    df_clustered,
    x='食品比率', y='高額品比率',
    color='cluster_label',
    size='商品数',
    hover_data=['shop_name', '主力カテゴリ'],
    title='自治体 食品比率 × 高額品比率',
    labels={'食品比率': '食品系商品の比率', '高額品比率': '3万円以上商品の比率'},
    color_discrete_sequence=px.colors.qualitative.Set1,
    opacity=0.7,
    height=500
)
fig.add_hline(y=0.3, line_dash='dot', line_color='gray')
fig.add_vline(x=0.5, line_dash='dot', line_color='gray')
fig.show()

In [ ]:
# 各クラスタの代表自治体（レビュー数合計TOP3）
for label in df_clustered['cluster_label'].unique():
    top3 = (
        df_clustered[df_clustered['cluster_label'] == label]
        .nlargest(3, 'レビュー数_合計')
        [['shop_name', '主力カテゴリ', '商品数', '価格_中央値', 'レビュー数_合計', '受入額_億円']]
    )
    print(f'\n【{label}】')
    print(top3.to_string(index=False))

## 5. カテゴリ内 自治体ポジショニング（詳細）

In [ ]:
# 牛肉カテゴリに絞った自治体間比較
from src.analysis.category import category_shop_analysis

shop_cat = category_shop_analysis(df)

for focus_cat in ['牛肉', '魚介類', '果物']:
    top = (
        shop_cat[shop_cat['category'] == focus_cat]
        .merge(df_clustered[['shop_name', 'cluster_label']], on='shop_name', how='left')
        .nlargest(20, 'レビュー数_合計')
    )
    fig = px.scatter(
        top,
        x='価格_中央値', y='レビュー数_合計',
        color='cluster_label',
        size='商品数',
        text='shop_name',
        title=f'【{focus_cat}】カテゴリ内 自治体ポジショニング（レビュー数TOP20）',
        labels={'価格_中央値': '価格中央値（円）', 'レビュー数_合計': 'レビュー数合計'},
        color_discrete_sequence=px.colors.qualitative.Set1,
        height=500
    )
    fig.update_traces(textposition='top center', marker=dict(sizemin=8))
    fig.show()

## 6. 分析結果の保存

In [ ]:
# parquet に保存
output_path = DATA_DIR / 'analysis/municipality_stats.parquet'
df_clustered.to_parquet(output_path, index=False)
print(f'保存完了: {output_path}')
print(f'\nクラスタ別 自治体数:')
print(df_clustered['cluster_label'].value_counts())

## 7. 分析まとめ・自治体戦略の示唆

### 5つの自治体タイプ
| タイプ | 特徴 | 代表的な戦略 |
|--------|------|----------|
| 食品コスパ型 | 低〜中価格、食品中心、レビュー多数 | 量×コスパで口コミを獲得 |
| 食品プレミアム型 | 中〜高価格、ブランド食品、評価高 | ブランド力で単価を維持 |
| 家電・高額品特化型 | 高価格、食品比率低 | 電化製品・高額品で大口寄付を獲得 |
| 大規模総合型 | 多品目・高受入額 | ポートフォリオ戦略で安定受入 |
| 汎用・小規模型 | 少商品数、低レビュー | まだ楽天出店の効果が出ていない |

### ECコンサルタント視点での示唆
- **「汎用・小規模型」がブルーオーシャン**: 楽天での存在感が薄く、商品ラインアップ整理と写真・説明文の改善だけで急成長できる可能性
- **「食品コスパ型」の成功パターン**: 訳あり商品・大容量パック・定期便のセットでレビューを増やし口コミを循環させている
- **「家電・高額品特化型」のリスク**: 商品ラインアップが偏るため、家電ブームが落ち着くと受入額が急落する可能性